# Azerbaijan DİM cutoff EDA and model comparison
This notebook imports the production training script. It reports the same split, features, baselines, and algorithms that write `models/metrics.json`; it does not maintain a second implementation.

The target is the next published cutoff at an Azerbaijani university, not a student's admission probability.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'backend').exists():
    sys.path.insert(0, str(ROOT / 'backend'))
else:
    sys.path.insert(0, str(ROOT.parent / 'backend'))
from scripts.train_cutoff_models import (
    evaluate_cold_start, evaluate_forecasting, load_country, models, prepare
)

data_dir = ROOT / 'data' / 'processed' if (ROOT / 'data').exists() else ROOT.parent / 'data' / 'processed'
df = prepare(load_country('AZ', data_dir))
print(f'{len(df):,} rows, {df.program_key.nunique():,} programmes, years {df.intake_year.min()}–{df.intake_year.max()}')

## DİM distribution

In [ ]:
display(df['cutoff_value'].describe(percentiles=[.1, .25, .5, .75, .9]).to_frame('cutoff'))
display(df.groupby('intake_year')['cutoff_value'].agg(['count', 'mean', 'median', 'std']))
df['cutoff_value'].plot(kind='hist', bins=30, title='Azerbaijan DİM cutoff distribution', xlabel='DİM cutoff')

## Production comparison: forecasting and cold start

In [ ]:
forecasting = evaluate_forecasting(df)
cold_start = evaluate_cold_start(df, ROOT / 'models', 'AZ')
print('FORECASTING')
display(pd.DataFrame(forecasting['baselines'] | forecasting['models']).T[['mae', 'rmse', 'r2', 'n']])
print(forecasting['split'])
print('COLD START')
display(pd.DataFrame(cold_start['baselines'] | cold_start['models']).T[['mae', 'rmse', 'r2', 'n']])

Both baselines are visible: global mean and department mean for cold start, persistence for forecasting. A negative `vs_baseline_pct` is a result, not a tuning invitation.

In [ ]:
# Show every production algorithm explicitly for both tasks.
for task_name, task in [('forecasting', forecasting), ('cold_start', cold_start)]:
    print(task_name, list(models()), {name: values['mae'] for name, values in task['models'].items()})

## Turkish investigation: failed remedies
These checks document why the strong persistence baseline is retained. They are deliberately evaluated without selecting a winner after looking at the answer.

In [ ]:
turkey = prepare(load_country('TR', data_dir))
train_end, _, test_year = int(turkey.intake_year.max()) - 2, int(turkey.intake_year.max()) - 1, int(turkey.intake_year.max())
usable = turkey.dropna(subset=['cut_lag1'])
train = usable[usable.intake_year <= train_end]
test = usable[usable.intake_year == test_year].copy()
persistence_mae = np.abs(test.cutoff_value - test.cut_lag1).mean()
delta_train = train.cutoff_value - train.cut_lag1
delta_prediction = np.repeat(delta_train.mean(), len(test)) + test.cut_lag1.to_numpy()
restricted = usable[usable.intake_year >= test_year - 2]
restricted_prediction = np.repeat(restricted.cutoff_value.mean(), len(test))
shrunken_prediction = 0.25 * test.cut_lag1.to_numpy() + 0.75 * train.cutoff_value.mean()
remedies = pd.DataFrame({
    'persistence': [persistence_mae],
    'delta_target': [np.abs(test.cutoff_value - delta_prediction).mean()],
    'restricted_window': [np.abs(test.cutoff_value - restricted_prediction).mean()],
    'shrinkage': [np.abs(test.cutoff_value - shrunken_prediction).mean()],
})
display(remedies.T.rename(columns={0: 'MAE'}))